# Paper Figure Reproduction

This notebook reproduces all main figures from the paper using pre-computed `eval_results.json` files.
No model inference is required — only the `results/` directory.

**Figures covered:**
- Fig 3: Coverage × Diversity heatmap (Section 4.2)
- Fig 10: SR vs Coverage (Appendix D.5)
- Fig 11: SR vs Diversity (Appendix D.5)
- Fig 2: SR vs Unique Questions (Section 4.1)
- Fig 4: Length scaling rescue (Section 5)
- Fig 5: RL spatial transfer (Section 6)
- Fig 8: Pretrain vs Finetune loss distribution (Appendix D.3)

In [ ]:
import numpy as np
import json
import os
import re
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.ticker import MaxNLocator

# Project root (one level up from notebooks/)
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
RESULTS_DIR = os.path.join(ROOT, 'results')
print(f'Results dir: {RESULTS_DIR}')
print(f'Exists: {os.path.isdir(RESULTS_DIR)}')

## Utility: Load eval results into matrices

In [ ]:
def load_cov_div_results(coverage_steps, diversity_range, result_dir, dataset_name='shortest_path', pairs_idx=0):
    """Load coverage-diversity grid results into matrices."""
    num_cov = len(coverage_steps)
    num_div = len(diversity_range)
    results_collect = defaultdict(dict)

    for i, cov in enumerate(coverage_steps):
        for j, div in enumerate(diversity_range):
            path = os.path.join(
                result_dir, 'cov_div',
                f'{dataset_name}/pretrain_random_walk_10M_reveal/'
                f'sft-shortest_path_reveal-diversity_{div}_coverage_{cov:.2f}_pairs_{pairs_idx}',
                'eval_results.json'
            )
            if not os.path.exists(path):
                continue
            try:
                with open(path) as f:
                    res = json.load(f)
                for idx, group in enumerate(res['G1_groups']):
                    group = tuple(group) if isinstance(group, list) else group
                    if group not in results_collect:
                        results_collect[group] = {
                            k: np.full((num_cov, num_div), np.nan)
                            for k in ['G1_valid_rates', 'G1_sp_rates', 'G2_valid_rates', 'G2_sp_rates']
                        }
                    for k in ['G1_valid_rates', 'G1_sp_rates', 'G2_valid_rates', 'G2_sp_rates']:
                        if k in res and res[k]:
                            results_collect[group][k][i, j] = res[k][idx]

                for agg, fn in [('mean', np.mean), ('max', np.max), ('min', np.min)]:
                    if agg not in results_collect:
                        results_collect[agg] = {
                            k: np.full((num_cov, num_div), np.nan)
                            for k in ['G1_valid_rates', 'G1_sp_rates', 'G2_valid_rates', 'G2_sp_rates']
                        }
                    for k in ['G1_sp_rates', 'G2_sp_rates', 'G1_valid_rates', 'G2_valid_rates']:
                        if k in res and res[k]:
                            results_collect[agg][k][i, j] = fn(res[k])
            except Exception as e:
                print(f'Error loading {path}: {e}')

    return results_collect


def load_qa_results(result_dir, coverages, ans_list, dataset_name='shortest_path'):
    """Load More Questions vs More Answers results."""
    results = {}
    for cov in coverages:
        for ans in ans_list:
            path = os.path.join(
                result_dir, 'spatial_length',
                f'{dataset_name}/pretrain_random_walk_10M_reveal/'
                f'sft-len20-shortest_path_reveal_coverage_{cov:.2f}_pairs_0_ans{ans}',
                'eval_results.json'
            )
            if os.path.exists(path):
                with open(path) as f:
                    results[(cov, ans)] = json.load(f)
    return results


def load_longshort_results(result_dir, dataset_name='shortest_path'):
    """Load length scaling rescue results."""
    base_dir = os.path.join(result_dir, 'spatial_length', f'{dataset_name}/longshort')
    results = {}
    if not os.path.isdir(base_dir):
        return results
    for d in os.listdir(base_dir):
        eval_path = os.path.join(base_dir, d, 'eval_results.json')
        if os.path.exists(eval_path):
            m = re.search(r'addnum_(\d+)_group_(.+)', d)
            if m:
                addnum = int(m.group(1))
                group = m.group(2)
                with open(eval_path) as f:
                    results[(addnum, group)] = json.load(f)
    return results


def load_grpo_results(result_dir, dataset_name='shortest_path'):
    """Load GRPO results."""
    base_dir = os.path.join(result_dir, 'spatial_length', f'{dataset_name}/pretrain_random_walk_10M_reveal')
    results = {}
    if not os.path.isdir(base_dir):
        return results
    for d in os.listdir(base_dir):
        if not d.startswith('grpo-'):
            continue
        grpo_dir = os.path.join(base_dir, d)
        if os.path.isdir(grpo_dir):
            for sub in os.listdir(grpo_dir):
                eval_path = os.path.join(grpo_dir, sub, 'eval_results.json')
                if os.path.exists(eval_path):
                    with open(eval_path) as f:
                        results[(d, sub)] = json.load(f)
    return results

## Figure 3: Coverage × Diversity Heatmap (Section 4.2)

In [ ]:
coverage_steps = [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.6, 0.8, 1.0]
diversity_range = [1, 2, 4, 8, 16, 32, 64, 128]

results_collect = load_cov_div_results(
    coverage_steps, diversity_range, RESULTS_DIR)

# Use mean across length groups for spatial transfer SR
G2_sp = np.array(results_collect['mean']['G2_sp_rates'])

# Labels
cov_labels = [f'{int(c*80)}%' for c in coverage_steps]  # 80% of nodes are in training region
div_labels = [f'$2^{i}$' for i in range(len(diversity_range))]

fig, ax = plt.subplots(1, 1, figsize=(6, 5))
h = sns.heatmap(G2_sp, ax=ax, cmap='YlGnBu', annot=True, fmt='.2f',
                xticklabels=div_labels, yticklabels=cov_labels)
ax.set_xlabel('Diversity Levels (Log Scale)', fontsize=12)
ax.set_ylabel('Coverage Levels (on Training Map G)', fontsize=12)
ax.set_title('Spatial Transfer SR', fontsize=13, fontweight='bold')

# Highlight efficient regime
ax.add_patch(Rectangle((2, 5), 4, 3.9, fill=False, edgecolor='red', linewidth=2, linestyle='--'))

plt.tight_layout()
plt.show()

## Figure 10: SR vs Coverage (Appendix D.5)

In [ ]:
norm = mcolors.LogNorm(vmin=min(diversity_range), vmax=max(diversity_range))
cmap = cm.get_cmap('coolwarm')

fig, ax = plt.subplots(figsize=(6, 4))
for j, div in enumerate(diversity_range):
    g2_data = results_collect[(1,30)]['G2_sp_rates'][:, j]
    ax.plot([c*0.8 for c in coverage_steps], g2_data,
            marker='o', color=cmap(norm(div)), label=f'{div}')

ax.set_xlabel('Coverage Levels (on Training Map G)', fontsize=12)
ax.set_ylabel('Spatial Transfer SR', fontsize=12)
ax.set_title('Spatial Transfer SR vs Coverage (Fig 10)', fontsize=13)
ax.set_ylim(0, 1.0)
ax.legend(title='Diversity', fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Figure 11: SR vs Diversity (Appendix D.5)

In [ ]:
cov_plot = [round(c*0.8, 3) for c in coverage_steps]
norm2 = mcolors.Normalize(vmin=min(cov_plot), vmax=max(cov_plot))
cmap2 = cm.get_cmap('coolwarm')

fig, ax = plt.subplots(figsize=(6, 4))
for i, cov in enumerate(cov_plot):
    if i == 0:
        continue
    g2_data = results_collect[(1,30)]['G2_sp_rates'][i, :]
    ax.plot(diversity_range, g2_data, marker='o',
            color=cmap2(norm2(cov)), label=f'{cov:.2f}')

ax.set_xlabel('Diversity Levels (Log Scale)', fontsize=12)
ax.set_ylabel('Spatial Transfer SR', fontsize=12)
ax.set_title('Spatial Transfer SR vs Diversity (Fig 11)', fontsize=13)
ax.set_xscale('log', base=2)
ax.set_ylim(0, 1.0)
ax.legend(title='Coverage', fontsize=9, bbox_to_anchor=(1.01, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Figure 2: SR vs Unique Questions (Section 4.1)

In [ ]:
coverages = [0.01, 0.05, 0.1, 0.2, 0.6, 0.8]
ans_list = [1, 2, 4, 8, 16, 32, 64]

qa_results = load_qa_results(RESULTS_DIR, coverages, ans_list)
print(f'Loaded {len(qa_results)} QA result files')

# For each coverage, compute avg spatial transfer SR across test maps
# Budget = coverage * total_pairs; #questions = budget / ans
budget_labels = {
    0.01: 'very low budget', 0.05: 'low budget', 0.1: 'less low budget',
    0.2: 'medium budget', 0.6: 'high budget', 0.8: 'very high budget'
}

fig, ax = plt.subplots(figsize=(7, 5))
for cov in coverages:
    x_vals, y_vals = [], []
    for ans in ans_list:
        if (cov, ans) in qa_results:
            res = qa_results[(cov, ans)]
            # Spatial transfer SR (G2, mean across length groups)
            sr = np.mean(res.get('G2_sp_rates', [0]))
            # Number of unique questions (approximate)
            n_questions = int(cov * 1600 * 1600 / ans)  # rough estimate
            x_vals.append(n_questions)
            y_vals.append(sr)
    if x_vals:
        ax.plot(x_vals, y_vals, marker='o', label=budget_labels.get(cov, f'cov={cov}'))

ax.set_xscale('log')
ax.set_xlabel('Number of Unique Questions (Bounded by Budget)', fontsize=12)
ax.set_ylabel('Spatial Transfer SR', fontsize=12)
ax.set_title('Spatial Transfer SR vs Unique Questions (Fig 2)', fontsize=13)
ax.axhline(y=0.97, color='gray', linestyle='--', alpha=0.5, label='SR = 0.97')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Figure 4: Length Scaling Rescue (Section 5)

In [ ]:
ls_results = load_longshort_results(RESULTS_DIR)
print(f'Loaded {len(ls_results)} longshort result files')

if ls_results:
    # Group by added length group, plot addnum vs SR at target length=30
    target_groups = {}
    for (addnum, group), res in sorted(ls_results.items()):
        if group not in target_groups:
            target_groups[group] = {'addnum': [], 'sr': []}
        # Get SR for the length group closest to 30
        sr = np.mean(res.get('G1_sp_rates', [0]))
        target_groups[group]['addnum'].append(addnum)
        target_groups[group]['sr'].append(sr)

    fig, ax = plt.subplots(figsize=(7, 5))
    for group in sorted(target_groups.keys()):
        data = target_groups[group]
        sort_idx = np.argsort(data['addnum'])
        ax.plot(
            np.array(data['addnum'])[sort_idx],
            np.array(data['sr'])[sort_idx],
            marker='o', label=f'l = {group}'
        )

    ax.set_xscale('log')
    ax.set_xlabel('Number of Added Training Paths', fontsize=12)
    ax.set_ylabel('Success Rate', fontsize=12)
    ax.set_title('Effect of Adding Paths on Length Scaling (Fig 4)', fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No longshort results found')

## Figure 5: RL Spatial Transfer (Section 6)

In [ ]:
grpo_results = load_grpo_results(RESULTS_DIR)
print(f'Loaded {len(grpo_results)} GRPO result files')

if grpo_results:
    # Parse: group by (coverage, num_generation) -> ckpt_idx -> SR
    parsed = defaultdict(dict)
    for (dirname, subdir), res in grpo_results.items():
        m_gen = re.search(r'num_generation_(\d+)', subdir)
        m_ckpt = re.search(r'from_ckpt_(\d+)', subdir)
        if m_gen and m_ckpt:
            n_gen = int(m_gen.group(1))
            ckpt = int(m_ckpt.group(1))
            sr = np.mean(res.get('G2_sp_rates', [0]))
            parsed[n_gen][ckpt] = sr

    fig, ax = plt.subplots(figsize=(7, 5))
    for n_gen in sorted(parsed.keys()):
        ckpts = sorted(parsed[n_gen].keys())
        srs = [parsed[n_gen][c] for c in ckpts]
        ax.bar([f'{c/1000:.0f}k' for c in ckpts], srs, alpha=0.7,
               label=f'rollout={n_gen}')

    ax.set_xlabel('SFT Checkpoint (RL Starting Stage)', fontsize=12)
    ax.set_ylabel('Spatial Transfer SR', fontsize=12)
    ax.set_title('RL Does Not Improve Spatial Transfer (Fig 5)', fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
else:
    print('No GRPO results found')

## Figure 8: Loss Distribution — Pretrain vs Finetune (Appendix D.3)

In [ ]:
pretrain_loss_file = os.path.join(RESULTS_DIR, 'hist_pretrain_loss.txt')
finetune_loss_file = os.path.join(RESULTS_DIR, 'hist_finetune_loss.txt')

if os.path.exists(pretrain_loss_file) and os.path.exists(finetune_loss_file):
    pre = np.loadtxt(pretrain_loss_file)
    ft = np.loadtxt(finetune_loss_file)
    pre = pre[np.isfinite(pre)]
    ft = ft[np.isfinite(ft)]

    fig, ax = plt.subplots(figsize=(6, 4))
    bins = 40
    xmin = min(pre.min(), ft.min())
    xmax = max(pre.max(), ft.max())

    ax.hist(ft, bins=bins, range=(xmin, xmax), density=True, alpha=0.6,
            color='tab:red', edgecolor='darkred', linewidth=0.8, label='Finetune')
    ax.hist(pre, bins=bins, range=(xmin, xmax), density=True, alpha=0.8,
            color='tab:blue', edgecolor='darkblue', linewidth=0.8, label='Pretrain')

    ax.set_xlabel('Per-sample Loss', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.set_title('Loss Distribution: Pretrain vs Finetune (Fig 8)', fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Loss histogram files not found in results/')